# Notebook 02 — Data Curation Pipeline

**Project**: ML-Based QSAR Modeling for Anti-Leishmanial Sulfonamide Derivatives  
**Author**: [Your Name]  
**Date**: May 2026  
**Purpose**: Clean and standardize the raw ChEMBL dataset following  
Fourches et al. (2010, 2016) curation guidelines.

**Input**: `data/raw/chembl_all_targets_raw.csv`  
**Output**:
- `data/processed/curated_dataset.csv`
- `data/processed/sulfonamide_subset.csv`
- `data/processed/train_set.csv`
- `data/processed/test_set.csv`

### Curation Pipeline:
```
Raw Data → SMILES Standardization → Remove Invalid → Handle Duplicates
         → IC50 → pIC50 → Activity Classification → Drug-likeness Filter
         → Train/Test Split → Curated Dataset
```

---


In [ ]:
# ============================================================
# CELL 1: Setup
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import sys
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem.MolStandardize import rdMolStandardize
from rdkit.Chem.Scaffolds import MurckoScaffold
from sklearn.model_selection import train_test_split

print(f"Project root: {PROJECT_ROOT}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M')}")


In [ ]:
# ============================================================
# CELL 2: Load Raw Data
# ============================================================
raw_path = PROJECT_ROOT / 'data' / 'raw' / 'chembl_all_targets_raw.csv'
df = pd.read_csv(raw_path)
print(f"Loaded raw data: {len(df)} records, {df['molecule_chembl_id'].nunique()} unique compounds")
print(f"Targets: {df['target_name'].unique().tolist()}")
print(f"\nShape: {df.shape}")
df.head()


## Step 1: SMILES Standardization\n\nFollowing Fourches et al. guidelines:\n- Remove salts and counterions (keep largest fragment)\n- Neutralize charges\n- Normalize tautomers\n- Canonicalize SMILES

In [ ]:
# ============================================================
# CELL 3: SMILES Standardization
# ============================================================

def standardize_smiles(smiles):
    """Standardize SMILES: remove salts, neutralize, normalize, canonicalize."""
    try:
        mol = Chem.MolFromSmiles(str(smiles))
        if mol is None:
            return None
        remover = rdMolStandardize.LargestFragmentChooser()
        mol = remover.choose(mol)
        uncharger = rdMolStandardize.Uncharger()
        mol = uncharger.uncharge(mol)
        normalizer = rdMolStandardize.Normalizer()
        mol = normalizer.normalize(mol)
        return Chem.MolToSmiles(mol, canonical=True)
    except Exception:
        return None

print("Standardizing SMILES (this may take a few minutes)...")
df['std_smiles'] = df['canonical_smiles'].apply(standardize_smiles)

n_before = len(df)
df = df.dropna(subset=['std_smiles'])
print(f"  Valid after standardization: {len(df)}/{n_before} ({n_before - len(df)} removed)")


## Step 2: Remove Invalid Activity Entries

In [ ]:
# ============================================================
# CELL 4: Remove Invalid Activities
# ============================================================
df['standard_value'] = pd.to_numeric(df['standard_value'], errors='coerce')
n_before = len(df)
df = df[df['standard_value'].notna()]
df = df[df['standard_value'] > 0]
print(f"After removing invalid activities: {len(df)}/{n_before}")


## Step 3: Handle Duplicates\n\n- **Concordant** (spread < 1 log unit): keep median\n- **Discordant** (spread ≥ 1 log unit): remove all

In [ ]:
# ============================================================
# CELL 5: Handle Duplicates
# ============================================================

grouped = df.groupby(['std_smiles', 'target_name'])
clean_rows = []
removed_discordant = 0
merged_concordant = 0

for (smi, target), group in grouped:
    if len(group) == 1:
        clean_rows.append(group.iloc[0])
    else:
        log_values = np.log10(group['standard_value'].values.astype(float))
        spread = log_values.max() - log_values.min()
        if spread < 1.0:
            median_row = group.iloc[0].copy()
            median_row['standard_value'] = group['standard_value'].median()
            clean_rows.append(median_row)
            merged_concordant += len(group) - 1
        else:
            removed_discordant += len(group)

df = pd.DataFrame(clean_rows).reset_index(drop=True)
print(f"Merged {merged_concordant} concordant duplicates")
print(f"Removed {removed_discordant} discordant entries")
print(f"After deduplication: {len(df)} entries")


## Step 4: Convert IC50 → pIC50

In [ ]:
# ============================================================
# CELL 6: Activity Conversion and Classification
# ============================================================

# Convert IC50 (nM) to pIC50 = -log10(IC50 in M)
df['pIC50'] = -np.log10(df['standard_value'].astype(float) * 1e-9)
print(f"pIC50 range: {df['pIC50'].min():.2f} – {df['pIC50'].max():.2f}")
print(f"pIC50 median: {df['pIC50'].median():.2f}")

# Classify: Active (pIC50 >= 5, i.e., IC50 <= 10 μM) vs Inactive
THRESHOLD = 5.0
df['activity_class'] = df['pIC50'].apply(lambda x: 'Active' if x >= THRESHOLD else 'Inactive')

n_active = (df['activity_class'] == 'Active').sum()
print(f"\nActive: {n_active} ({n_active/len(df)*100:.1f}%)")
print(f"Inactive: {len(df) - n_active} ({(len(df)-n_active)/len(df)*100:.1f}%)")


## Step 5: Drug-likeness Filter

In [ ]:
# ============================================================
# CELL 7: Drug-likeness Filter
# ============================================================
metals = {'Fe', 'Cu', 'Zn', 'Mn', 'Co', 'Ni', 'Pt', 'Pd', 'Ru',
          'Rh', 'Ir', 'Os', 'Au', 'Ag', 'Hg', 'Cd', 'Cr', 'Mo', 'W'}

keep_mask = []
for _, row in df.iterrows():
    mol = Chem.MolFromSmiles(str(row['std_smiles']))
    if mol is None:
        keep_mask.append(False)
        continue
    mw = Descriptors.MolWt(mol)
    if mw > 900:
        keep_mask.append(False)
        continue
    atom_symbols = {atom.GetSymbol() for atom in mol.GetAtoms()}
    if atom_symbols & metals:
        keep_mask.append(False)
        continue
    keep_mask.append(True)

df = df[keep_mask].reset_index(drop=True)
print(f"After drug-likeness filter: {len(df)} compounds")


## Step 6: Add Scaffolds and Split Dataset

In [ ]:
# ============================================================
# CELL 8: Murcko Scaffolds + Train/Test Split
# ============================================================

# Compute Murcko scaffolds
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return ''
    scaffold = MurckoScaffold.GetScaffoldForMol(mol)
    return Chem.MolToSmiles(scaffold)

print("Computing Murcko scaffolds...")
df['scaffold'] = df['std_smiles'].apply(get_scaffold)
print(f"Unique scaffolds: {df['scaffold'].nunique()}")

# Stratified random split (70/30)
train_df, test_df = train_test_split(
    df, test_size=0.30, random_state=42,
    stratify=df['activity_class']
)
print(f"\nTraining set: {len(train_df)} compounds")
print(f"Test set: {len(test_df)} compounds")
print(f"Train active ratio: {(train_df['activity_class'] == 'Active').mean():.1%}")
print(f"Test active ratio: {(test_df['activity_class'] == 'Active').mean():.1%}")


In [ ]:
# ============================================================
# CELL 9: Save Curated Datasets
# ============================================================
processed_dir = PROJECT_ROOT / 'data' / 'processed'
processed_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(processed_dir / 'curated_dataset.csv', index=False)
train_df.to_csv(processed_dir / 'train_set.csv', index=False)
test_df.to_csv(processed_dir / 'test_set.csv', index=False)

if 'is_sulfonamide' in df.columns:
    sulfo = df[df['is_sulfonamide']]
    sulfo.to_csv(processed_dir / 'sulfonamide_subset.csv', index=False)
    print(f"Sulfonamide subset: {len(sulfo)} compounds")

print("\nAll curated files saved:")
for f in sorted(processed_dir.glob('*.csv')):
    print(f"  ✓ {f.name}")


In [ ]:
# ============================================================
# CELL 10: Visualization — Curated Data Summary
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. pIC50 distribution
axes[0,0].hist(df['pIC50'], bins=40, color='#2563EB', alpha=0.7, edgecolor='white')
axes[0,0].axvline(x=5.0, color='red', linestyle='--', linewidth=1.5, label='Active threshold')
axes[0,0].set_xlabel('pIC50')
axes[0,0].set_ylabel('Count')
axes[0,0].set_title('pIC50 Distribution (Curated)')
axes[0,0].legend()

# 2. Class balance by target
class_counts = df.groupby(['target_name', 'activity_class']).size().unstack(fill_value=0)
class_counts.plot(kind='bar', ax=axes[0,1], color=['#DC2626', '#16A34A'])
axes[0,1].set_title('Active vs Inactive by Target')
axes[0,1].set_ylabel('Count')
axes[0,1].tick_params(axis='x', rotation=45)

# 3. Activity type distribution
df['standard_type'].value_counts().plot(kind='bar', ax=axes[1,0], color='#8B5CF6')
axes[1,0].set_title('Activity Type Distribution')
axes[1,0].set_ylabel('Count')

# 4. Molecular weight distribution
mw_values = []
for smi in df['std_smiles']:
    mol = Chem.MolFromSmiles(str(smi))
    if mol:
        mw_values.append(Descriptors.MolWt(mol))
axes[1,1].hist(mw_values, bins=40, color='#F59E0B', alpha=0.7, edgecolor='white')
axes[1,1].set_xlabel('Molecular Weight (Da)')
axes[1,1].set_ylabel('Count')
axes[1,1].set_title('Molecular Weight Distribution')

plt.suptitle('Curated Dataset Summary', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'figures' / 'curated_data_summary.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Curation complete! Proceed to Notebook 03: Descriptor Calculation")
